In [6]:
import sqlite3
import pandas as pd
from collections import Counter, defaultdict
from itertools import combinations
import random
import math

# ============================================================
# 1. LOAD DATA
# ============================================================
sqlite_path = r"D:\Tananyagok\DATA SCIENCE MSC\DSLab\data\ocel2-p2p.sqlite"
conn = sqlite3.connect(sqlite_path)

event_object = pd.read_sql("""
SELECT ocel_event_id, ocel_object_id
FROM event_object
""", conn)

conn.close()

# ============================================================
# 2. BUILD EVENT STRUCTURE
# ============================================================
# Map each event to its set of associated objects: O(e)
event_objects = event_object.groupby("ocel_event_id")["ocel_object_id"].apply(set)

event_df = pd.DataFrame({
    "event_id": event_objects.index,
    "objects": event_objects.values
}).reset_index(drop=True)

print(f"Total events loaded: {len(event_df)}")

# ============================================================
# 3. TRAIN / TEST SPLIT
# ============================================================
random.seed(42)

train_df = event_df.sample(frac=0.7, random_state=42)
test_df = event_df.drop(train_df.index)

# ============================================================
# 4. COMPUTE GLOBAL SUPPORT & CONDITIONAL PROBABILITIES
# ============================================================
support_singleton = Counter()
support_pair = Counter()

# Count occurrences of singletons and pairs in the training set
for objs in train_df["objects"]:
    for o in objs:
        support_singleton[o] += 1
    for o1, o2 in combinations(objs, 2):
        support_pair[(o1, o2)] += 1
        support_pair[(o2, o1)] += 1

# Calculate \hat{P}(o_2 | o_1) = supp({o1, o2}) / supp({o1})
# We store this as a nested dictionary: conditional_prob[o1][o2] = prob
conditional_prob = defaultdict(dict)

for (o1, o2), pair_supp in support_pair.items():
    sing_supp = support_singleton[o1]
    if sing_supp > 0:
        conditional_prob[o1][o2] = pair_supp / sing_supp

# ============================================================
# 5. CORRUPTION MODEL (Simulate Partially Observed Logs)
# ============================================================
def corrupt(objs, drop_fraction=0.3):
    objs_list = list(objs)
    if len(objs_list) <= 1:
        return set(objs_list)
    k = max(1, int(len(objs_list) * drop_fraction))
    return set(objs_list) - set(random.sample(objs_list, k))

test_df["corrupted"] = test_df["objects"].apply(corrupt)

# ============================================================
# 6. SCORING AND RECONSTRUCTION (PREDICTION)
# ============================================================
def predict(observed_objs, tau=-15.0):
    """
    Reconstructs missing objects using log-probability aggregation.
    
    1. Generates candidates: C(e) = union of { o' | P(o' | o) > 0 } for o in O(e)
    2. Scores candidates: s(o') = sum_{o in O(e)} log(P(o' | o))
    3. Filters by threshold: R(e) = { o' in C(e) | s(o') >= tau }
    """
    # Step 1: Generate candidate pool C(e)
    candidates = set()
    for o in observed_objs:
        for o_prime in conditional_prob[o].keys():
            if o_prime not in observed_objs:
                candidates.add(o_prime)
                
    # Step 2: Assign scores s(o')
    scores = {}
    for o_prime in candidates:
        total_log_prob = 0.0
        for o in observed_objs:
            # Estimate probability with a small epsilon to smooth unobserved pairs
            prob = conditional_prob[o].get(o_prime, 0.0)
            total_log_prob += math.log(prob + 1e-9)
        scores[o_prime] = total_log_prob
        
    # Step 3: Reconstruction R(e) based on threshold tau
    reconstructed = {o_prime for o_prime, score in scores.items() if score >= tau}
    return reconstructed

# ============================================================
# 7. EVALUATION
# ============================================================
def evaluate(df, tau=-15.0, num_samples=3):
    total_missing = 0
    correct = 0
    total_predicted = 0
    samples_printed = 0

    for _, row in df.iterrows():
        true_set = row["objects"]
        observed = row["corrupted"]

        missing = true_set - observed
        if not missing:
            continue

        preds = predict(observed, tau=tau)

        total_missing += len(missing)
        total_predicted += len(preds)

        for m in missing:
            if m in preds:
                correct += 1

        # Print sample event details during evaluation
        if samples_printed < num_samples:
            print("--- SAMPLE EVENT ---")
            print(f"True: {true_set}")
            print(f"Observed: {observed}")
            print(f"Predicted: {list(preds)}")
            samples_printed += 1

    recall = correct / total_missing if total_missing else 0
    precision = correct / total_predicted if total_predicted else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) else 0

    return recall, precision, f1

# Run evaluation with a default threshold
recall, precision, f1 = evaluate(test_df, tau=-50.0)

print("\n=== FINAL EVALUATION ===")
print(f"Recall:    {recall:.4f}")
print(f"Precision: {precision:.4f}")
print(f"F1 Score:  {f1:.4f}")

Total events loaded: 14671
--- SAMPLE EVENT ---
True: {'purchase_order:8', 'quotation:3'}
Observed: {'quotation:3'}
Predicted: ['purchase_order:7', 'purchase_requisition:5:pr_trigger_5']
--- SAMPLE EVENT ---
True: {'purchase_order:537', 'goods receipt:645'}
Observed: {'goods receipt:645'}
Predicted: ['goods receipt:646', 'purchase_order:537', 'purchase_order:536', 'goods receipt:644', 'payment:317', 'invoice receipt:651', 'purchase_order:538', 'goods receipt:647']
--- SAMPLE EVENT ---
True: {'invoice receipt:640', 'goods receipt:639'}
Observed: {'invoice receipt:640'}
Predicted: ['purchase_order:528', 'goods receipt:639', 'payment:310']

=== FINAL EVALUATION ===
Recall:    0.6691
Precision: 0.1831
F1 Score:  0.2875
